## Great, now that we discussed a little let's continue

Given that the current approach utilized by the authors lacks reproducibility, we will explore an alternative method by leveraging nf-core pipelines for data analysis.

Please explain, how we will achieve reproducibility for the course  with this approach.


1. Use of fixed pipeline versions: Run each pipeline with a specific release (e.g. `-r 2.0.0`).So everyone uses exactly the same code, even if the pipeline is updated late

2. Use of Containers (Docker): All tools run inside containers with fixed software versions.The results do not depend on what is installed on each computer.

3. Same inputs and parameters:Use same public data, same refernce genome and the same samplesheets

4. Automatic documentation: Nextflow and nf-core save the software versions, parameters and run reports so every run can be traced and repeated.

5. Community standards: nf-core pipelines follow strict guidelines, are tested and reviewed, and use best-practice tools. This makes the analysis transparent.

6. Portability: The same pipeline runs on a laptop, a cluster or in the cloud and gives the same results.


You have successfully downloaded 2 of the fastq files we will use in our study.

What is the next step if we want to first have a count table and check the quality of our fastq files? What is the pipeline called to do so?

The next step is to run nf-core/rnaseq run with HISAT2 as aligner.

This pipeline checks the quality of the raw reads (FastQC, MultiQC), trims adapters and low-quality bases, quantifies the reads against the mouse reference and creates a gene count table for all samples. The count table can then be used for differential expression analysis with nf-core/differentialabundance.

Analyze the 2 files using an nf-core pipeline.

### What does this pipeline do?

nf-core/rnaseq analyses RNA-seq data. It takes the raw FASTQ files and:
1. checks the quality of the raw reads
2. trims adapters and low-quality bases
3. aligns the reads to the mouse reference genome
4. checks the quality of the alignments
5. quantifies gene expression and creates a gene count table for all samples
6. summarizes all quality metrics in one report

### Which are the main tools that will be used in the pipeline?

- FastQC: quality control of the raw reads
- Trim Galore (Cutadapt):** trimming of adapters and low-quality bases
- HISAT2: alignment of the reads to the genome
- SAMtools / Picard:** sorting, indexing and marking duplicates in the alignment files
- RSeQC, Qualimap, Preseq, dupRadar:** quality control of the alignments
- kallisto: pseudo-alignment and quantification, produces the gene count table
- MultiQC: one summary report of all quality metrics

As all other nf-core pipelines, the chosen pipeline takes in a samplesheet as input.

Use Python and pandas to create the samplesheet for your 2 samples. Feel free to make use of the table you created earlier today.

Choose your sample names wisely, they must be the connection of the results to the metadata. If you can't find the sample in the metadata later, the analysis was useless.

In [3]:
#create samplesheet and add strandedness

import pandas as pd
sheet = pd.read_csv('fetchngs_results/samplesheet/samplesheet.csv')
new = sheet[['sample', 'fastq_1', 'fastq_2']].copy()
new['strandedness'] = 'auto'
new.to_csv('samplesheet_rnaseq.csv', index=False)
print(new)


        sample                                            fastq_1  \
0  SRX19144486  fetchngs_results/fastq/SRX19144486_SRR23195516...   
1  SRX19144488  fetchngs_results/fastq/SRX19144488_SRR23195511...   

                                             fastq_2 strandedness  
0  fetchngs_results/fastq/SRX19144486_SRR23195516...         auto  
1  fetchngs_results/fastq/SRX19144488_SRR23195511...         auto  


In [12]:
# post here the command you used to run nf-core/rnaseq
!nextflow run nf-core/rnaseq -r 3.27.0 -profile docker -c local.config \
    --input samplesheet_rnaseq.csv \
    --outdir results \
    --fasta https://ftp.ensembl.org/pub/release-113/fasta/mus_musculus/dna/Mus_musculus.GRCm39.dna.primary_assembly.fa.gz \
    --gtf https://ftp.ensembl.org/pub/release-113/gtf/mus_musculus/Mus_musculus.GRCm39.113.gtf.gz \
    --aligner hisat2 \
    --pseudo_aligner kallisto \
    --skip_alignment \
    -resume
    


 N E X T F L O W   ~  version 26.04.6

Launching `https://github.com/nf-core/rnaseq` [happy_poisson] revision: a1fcdddd3b [3.27.0]

WARN: Unrecognized config option 'manifest.diagram'

------------------------------------------------------
                                        ,--./,-.
        ___     __   __   __   ___     /,-._.--~'
  |\ | |__  __ /  ` /  \ |__) |__         }  {
  | \| |       \__, \__/ |  \ |___     \`-._,-`-,
                                        `._,._,'
  nf-core/rnaseq 3.27.0
------------------------------------------------------

Input/output options
  input              : samplesheet_rnaseq.csv
  outdir             : results

Reference genome options
  fasta              : https://ftp.ensembl.org/pub/release-113/fasta/mus_musculus/dna/Mus_musculus.GRCm39.dna.primary_assembly.fa.gz
  gtf                : https://ftp.ensembl.org/pub/release-113/gtf/mus_musculus/Mus_musculus.GRCm39.113.gtf.gz

UMI options
  umi_discard_read   : 0

Alignment options
  aligner

Explain all the parameters you set and why you set them in this way.



| Parameter | Explanation |
|---|---|
| `nextflow run nf-core/rnaseq` | Runs the nf-core/rnaseq pipeline |
| `-r 3.27.0` | Uses a fixed pipeline version, so the analysis is reproducible |
| `-profile docker` | Runs all tools in Docker containers with fixed software versions |
| `--input` | Samplesheet with sample names, paths to the FASTQ files and strandedness |
| `--outdir` | Folder where all results are stored |
| `--fasta` | Reference genome: contains the raw DNA sequence |
| `--gtf` | Gene annotation: contains the positions of genes, transcripts and exons |
| `--aligner hisat2` | The authors used HISAT2 for alignment, so our results are easier to compare with the original study. It also needs much less RAM than STAR. But alignement is skipped due to memory issues |
| `--pseudo_aligner kallisto` | Matches reads directly to transcripts instead of aligning them base by base to the genome. This is much faster, needs little RAM and produces the gene count table |
| `--skip_alignment` | For differential expression we only need to know how many reads come from each gene. Pseudo-aligners like kallisto estimate this directly with similar accuracy, much faster and with much less memory |

## Browsing the results

How did the pipeline perform?

The further analysis was done wit the provided data (Star alignment + Salmon and whole refence genome)
(STAR alignment + Salmon quantification, mouse reference GRCm38).
Each sample has about 41–55 million read pairs. Most samples look good, but a few samples show clear quality problems (see below).

In [13]:
import pandas as pd

folder = "rnaseq_out_allsamples/multiqc/star_salmon/multiqc_report_data/"

# general statistics (one row per sample)
stats = pd.read_csv(folder + "multiqc_general_stats.txt", sep="\t")
stats = stats[stats["Sample"].str.match(r"^SRX\d+$")].set_index("Sample")

qc = pd.DataFrame({
    "reads_million": stats["fastqc_raw-total_sequences"],
    "mapped_%": stats["star-mapped_percent"],
    "duplication_%": stats["picard_mark_duplicates-PERCENT_DUPLICATION"],
    "GC_%": stats["fastqc_raw-percent_gc"],
})

# rRNA from the biotype counts
biotypes = pd.read_csv(folder + "multiqc_featurecounts_biotype_plot.txt", sep="\t").set_index("Sample")
total = biotypes.sum(axis=1)
qc["rRNA_%"] = 100 * biotypes["rRNA"] / total
qc["Mt_rRNA_%"] = 100 * biotypes["Mt_rRNA"] / total

# strandedness check
strand = pd.read_csv(folder + "multiqc_strand_check_summary_table.txt", sep="\t").set_index("Sample")
qc["strand_check"] = strand["status"]

qc.round(1)

,reads_million,mapped_%,duplication_%,GC_%,rRNA_%,Mt_rRNA_%,strand_check
Sample,,,,,,,
SRX19144479,52.4,85.2,68.0,51.5,3.3,6.4,pass
SRX19144480,46.6,83.7,58.8,49.5,0.1,1.7,pass
SRX19144481,52.7,82.9,81.4,56.0,24.4,17.5,fail
SRX19144482,45.8,89.6,44.2,47.5,0.1,5.1,pass
SRX19144483,54.5,92.4,59.6,49.5,0.0,5.3,pass
SRX19144484,48.2,82.0,50.5,53.5,9.3,1.7,fail
SRX19144485,54.0,73.6,23.1,51.0,0.2,4.2,pass
SRX19144486,41.4,91.0,71.8,48.0,0.0,2.3,pass
SRX19144487,49.8,94.6,31.2,48.5,0.0,5.4,pass


Explain the quality control steps. Are you happy with the quality and why. If not, why not.
Please give additional information on : 
- ribosomal rRNA
- Duplication
- GC content

What are the possible steps that could lead to poorer results?

#### Explain quality control steps

- FastQC: checks base quality, adapter content, GC content and duplication of the raw reads
- Trim Galore: removes adapters and low-quality bases, followed by a second FastQC
- HISAT2 alignment stats and RSeQC/Qualimap: how many reads map, where they map (exons, introns, intergenic) and the gene body coverage
- MultiQC: summarizes everything in one report

#### Evaluating the results

Most samples have good quality: 82–95 % of the reads map to the genome, rRNA is below 1 %,the GC content is normal (47–53 %), the library is reverse-stranded, and the samples cluster together
in the PCA. 

However, some samples show problems:

Ribosomal RNA: 
rRNA should be removed during library preparation. Most samples have < 1 % rRNA,but SRX19144481 (24 % + 18 % mitochondrial rRNA), SRX19144489 (21 %) and SRX19144484 (9 %) contain a lot of rRNA, so the rRNA depletion / poly-A selection did not work well.


Duplication:
Duplication is between 23 % and 88 %. Some duplication is normal in RNA-seq,because highly expressed genes produce many identical reads. Very high values in SRX19144489 (88 %) and SRX19144481 (81 %) are caused by the high rRNA content and point to low library complexity.

GC content:
Most samples have 47–53 % GC, as expected for mouse. SRX19144492 has 63 % GC and only 28 % of its reads map to the mouse genome, which suggests contamination (e.g. bacteria or adapter dimers). SRX19144481, SRX19144489 and SRX19144491 also have a slightly higher GC content (55–56 %).

Possible reasons for poorer results:
RNA degradation, low input amount, failed rRNA depletion,
contamination, too many PCR cycles, adapter dimers, or technical batch effects during library preparation.

#### Additional Information 

##### Ribosomal DNA

Ribosomal RNA (rRNA): rRNA makes up most of the RNA in a cell. Library preparation
(poly-A selection or rRNA depletion) should remove it. A high rRNA fraction means that
the depletion did not work well and fewer reads are left for the mRNAs of interest.

##### Duplication

In RNA-seq, some duplication is normal. Identical reads can be biological, because highly expressed genes produce many reads that start at the same position by chance, or technical, because PCR creates copies of the same fragment. dupRadar helps to tell them apart.

##### GC Content

The GC distribution of the reads should look like a single, smooth peak
that matches the mouse transcriptome. Extra peaks or a shifted curve can point to contamination (e.g. bacteria, adapter dimers) or PCR bias.

#### What are the possible steps that could lead to poorer results?

Sequencing:
- Low sequencing depth, so lowly expressed genes are missed
- Low base quality towards the end of the reads or problems with the flow cell
- Uneven pooling, so some samples get far fewer reads than others

Libary Preparation: 
-Failed rRNA depletion or poly-A selection, which leaves a high rRNA fraction
- Too many PCR cycles, which create many PCR duplicates and a GC bias
- Adapter dimers or short fragments (high adapter content)
- Samples prepared on different days or with different kits (batch effects)

Would you exclude any samples? If yes, which and why?

Yes. I would exclude SRX19144489 (strong outlier in the PCA, 21 % rRNA, 88 % duplication, only 65 % mapped reads, failed strandedness check), SRX19144481 (42 % rRNA, 81 % duplication, failed strandedness check) and SRX19144492 (only 28 % mapped reads and 63 % GC maybe caused by contamination).

What would you now do to continue the experiment? What are the scientists trying to figure out? Which packages on R or python would you use?

To continue the experiment, I would:
1. Download all 16 samples with nf-core/fetchngs and process them with nf-core/rnaseq,
   not only the 2 test samples.
2. Check the quality of all samples in the MultiQC report and exclude samples with poor quality
   (e.g. high rRNA content, low mapping rate or contamination).
3. Run nf-core/differentialabundance with the gene count table, a samplesheet with the metadata
   (genotype and treatment) and a contrasts file to compare the groups,
   e.g. Sham-Oxy vs Sham-Sal, SNI-Sal vs Sham-Sal, SNI-Oxy vs SNI-Sal.
4. Perform pathway / gene set enrichment analysis on the differentially expressed genes to find out which biological processes and regulators are affected.

THe scientist trying to figure out which genes and pathways change in the brain's reward system during oxycodone withdrawal,
how chronic pain changes this response, and whether HDAC1/HDAC2 is a key regulator
that could be targeted with a drug.

Packages: 

- R: DESeq2 or edgeR/limma (differential expression), tximport (import of kallisto counts),
  clusterProfiler or fgsea (pathway / gene set enrichment), ggplot2, EnhancedVolcano, pheatmap (plots)
- Python: pandas (tables), PyDESeq2 (differential expression), GSEApy (enrichment),
  matplotlib / seaborn (plots)